# Section 3 - Generation + Evaluation

Section 2 built the retriever. This section finishes the RAG pipeline from Section 1.6, then checks how well the retriever works:

1. **Generation:** paste the retrieved modules into the prompt (step 3, *augment*) and let Gemini answer from them (step 4, *generate*).
2. **Evaluation:** measure how often the retriever finds the right module, because the answer can only be as good as what was retrieved.

In [ ]:
# Standalone setup, for working on Section 3 on its own.
# scripts/build_workshop.py drops this cell: Sections 1 and 2 already do all of it.
# %pip install -q google-genai sentence-transformers numpy pandas
import re
import sys
import textwrap

sys.path.append("..")  # rag_utils.py is one folder up from notebooks/

from IPython.display import Markdown, display

import rag_utils
from rag_utils import ask_gemini, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
rag_utils.connect_gemini()

# From Section 1.5
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS4248, CS2103T


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")


# From Section 2
def show(results):
    """Print (module, score) pairs, one per line."""
    for module, score in results:
        print(f"  {score:.2f}  {module['moduleCode']:<9} {module['title']}")

## 3.1 Generation

Two small functions do the augment step:

- `make_context` turns the `(module, score)` pairs from `search` into plain text, one numbered block per module.
- `build_prompt` wraps that text in instructions: answer **only** from the context, say so when the context is not enough, and cite module codes in square brackets.

In [ ]:
def make_context(results):
    """Format retrieved (module, score) pairs as numbered text blocks."""
    blocks = []
    for rank, (module, score) in enumerate(results, start=1):
        lines = [
            f"[{rank}] {module['moduleCode']} - {module['title']}",
            f"Description: {module['description']}",
        ]
        workload = rag_utils.format_workload(module.get("workload"))
        if workload:
            lines.append(f"Workload: {workload}")
        if module.get("semesters"):
            lines.append(f"Semesters offered in AY {rag_utils.AY}: {', '.join(map(str, module['semesters']))}")
        if module.get("hasExam") is not None:
            lines.append(f"Has final exam: {'yes' if module['hasExam'] else 'no'}")
        lines.append(f"Retrieval similarity: {score:.3f}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks) if blocks else "No modules were retrieved."


def build_prompt(question, context):
    """Wrap the question and the retrieved context in grounding instructions."""
    return f"""You are a helpful assistant answering questions about NUS modules.
Answer the user's question using only the supplied NUSMods context.

Rules:
- If the context does not contain enough evidence, say that the information is not available in the supplied context.
- Do not invent module codes, titles, prerequisites, workload hours, semesters, or exam details.
- When making a claim about a module, cite its module code in square brackets, e.g. [CS1231S].
- Keep the answer concise and distinguish explicit facts from uncertainty.
- The retrieval similarity score is not evidence that a claim is true.

Retrieved context:
{context}

User question: {question}

Answer:"""


def rag_answer(question, k=5, results=None, show_prompt=False):
    """Retrieve, augment, generate. Pass `results` to supply your own context."""
    if results is None:
        results = search(question, k=k)
    prompt = build_prompt(question, make_context(results))
    if show_prompt:
        print(prompt)
        print("-" * 60)
    answer = ask_gemini(prompt)
    display(Markdown(answer))
    return answer

### The same question, now with RAG

In Section 1.5, Gemini answered this question from memory, and `verify` caught the module codes it got wrong. Here is the exact prompt RAG sends for the same question. Gemini sees only this text.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

print(build_prompt(question, make_context(search(question, k=5))))

In [ ]:
answer = rag_answer(question)
print("-" * 60)
verify(answer)

Every module code Gemini can cite now comes from the context, so `verify` should find each one in the real data. Compare this with the 1.5 result.

That fixes invented codes, but not missing ones. Look back at the retrieved modules in the prompt: are they about attention in neural networks, or attention in psychology? If the right module was not retrieved, Gemini cannot mention it. The answer is only as good as the context.

### Wrong context, confident answer

To see how much the answer depends on retrieval, keep the question and swap in modules retrieved for an unrelated topic.

In [ ]:
wrong_results = search("modules about economics and financial markets", k=5)
show(wrong_results)
print("-" * 60)

wrong_answer = rag_answer(question, results=wrong_results)

**How to read it:** the prompt tells Gemini to say when the context is not enough. Check whether it did, or whether it stretched an economics module to fit a question about attention mechanisms. Either way, generation cannot recover from bad retrieval, which is why 3.2 measures the retriever on its own.

**YOUR TURN:** ask your own question. Set `show_prompt=True` to see exactly what Gemini received, and change `k` to give it more or fewer modules.

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules teach reinforcement learning? List the module codes and titles."

my_answer = rag_answer(my_question, k=5, show_prompt=False)
print("-" * 60)
verify(my_answer)

## 3.2 Evaluation

Now we evaluate how effectively our RAG system retrieves the correct NUS module for a user's question.

For each query, we specify the expected module code and compare it with the modules returned by our retrieval system.

We use two metrics:

- **Hit@5:** Measures the proportion of queries for which the correct module appears in the top five retrieved results.
- **Mean Reciprocal Rank (MRR):** Measures how highly the correct module is ranked, rewarding results that appear closer to the top.

These metrics help us assess retrieval performance before investigating potential improvements.

In [ ]:
import pandas as pd

workshop_dir = rag_utils.DATA_DIR.parent  # the week-11-nlp folder

In [ ]:
eval_path = workshop_dir / "data" / "eval_set_v2.csv"

eval_df = pd.read_csv(eval_path)

print(f"Total evaluation queries: {len(eval_df)}")
print("\nQueries by category:")
display(eval_df["category"].value_counts().to_frame("count"))

display(eval_df.head())

In [ ]:
def evaluate_multilabel_retrieval(eval_df, k=5):
    details = []

    for _, row in eval_df.iterrows():
        query = str(row["query"])

        relevant_codes = {
            code.strip().upper()
            for code in str(row["relevant"]).split("|")
            if code.strip()
        }

        results = rag_utils.search(query, k=k)

        ranked_codes = [
            str(module.get("moduleCode", "")).upper()
            for module, score in results
        ]

        # Find ranks of retrieved relevant modules
        relevant_ranks = [
            rank
            for rank, code in enumerate(ranked_codes, start=1)
            if code in relevant_codes
        ]

        first_relevant_rank = (
            min(relevant_ranks) if relevant_ranks else 0
        )

        # Hit@k: at least one relevant module appears in top k
        hit_at_k = int(bool(relevant_ranks))

        # Recall@k: fraction of all relevant modules retrieved in top k
        recall_at_k = (
            len(set(ranked_codes) & relevant_codes) / len(relevant_codes)
            if relevant_codes else 0
        )

        # Reciprocal rank of the first relevant result
        reciprocal_rank = (
            1 / first_relevant_rank
            if first_relevant_rank else 0
        )

        details.append({
            "query": query,
            "category": row["category"],
            "relevant_codes": " | ".join(sorted(relevant_codes)),
            "top_k_module_codes": " | ".join(ranked_codes),
            "rank_of_first_relevant": first_relevant_rank,
            f"hit_at_{k}": hit_at_k,
            f"recall_at_{k}": recall_at_k,
            "reciprocal_rank": reciprocal_rank
        })

    details_df = pd.DataFrame(details)

    summary = {
        "n_queries": len(details_df),
        f"hit_at_{k}": details_df[f"hit_at_{k}"].mean(),
        f"recall_at_{k}": details_df[f"recall_at_{k}"].mean(),
        "mrr": details_df["reciprocal_rank"].mean()
    }

    return summary, details_df

In [ ]:
summary, details_df = evaluate_multilabel_retrieval(
    eval_df,
    k=5
)

print(f"Queries evaluated: {summary['n_queries']}")
print(f"Hit@5: {summary['hit_at_5']:.3f}")
print(f"Recall@5: {summary['recall_at_5']:.3f}")
print(f"MRR: {summary['mrr']:.3f}")

results_dir = workshop_dir / "evaluation"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "eval_set_v2_results.csv"
details_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

display(details_df)

In [ ]:
import pandas as pd
from IPython.display import display

# Class-wise evaluation
classwise_summary = (
    details_df
    .groupby("category")
    .agg(
        queries=("query", "count"),
        hit_at_5=("hit_at_5", "mean"),
        recall_at_5=("recall_at_5", "mean"),
        mrr=("reciprocal_rank", "mean")
    )
    .reset_index()
)

# Convert metrics to percentages where appropriate
classwise_summary["hit_at_5"] *= 100
classwise_summary["recall_at_5"] *= 100

# Round results
classwise_summary = classwise_summary.round({
    "hit_at_5": 2,
    "recall_at_5": 2,
    "mrr": 3
})

print("Class-wise Retrieval Evaluation Results")
display(classwise_summary)


## Interpretation of Semantic Retrieval Evaluation Results


- **Keyword queries:** Achieved a Hit@5 of 100%, indicating that at least one relevant module was retrieved within the top five results for every query. Recall@5 was 67.50%, and MRR was 0.670, showing that the system retrieved a substantial proportion of relevant modules and generally ranked relevant results highly.

- **Paraphrase queries:** Achieved a Hit@5 of 60% and Recall@5 of 40%. This indicates that the system can retrieve relevant modules for queries expressed using different wording, although some relevant modules are missed or ranked outside the top five.

- **Prerequisite queries:** Achieved a Hit@5 of 60%, Recall@5 of 34.17%, and MRR of 0.425. The results suggest that semantic retrieval can identify modules related to prerequisite concepts, but retrieving all relevant modules remains challenging.

- **Vague queries:** Achieved the lowest performance, with a Hit@5 of 20%, Recall@5 of 15%, and MRR of 0.200. This suggests that queries lacking specific terms or sufficient context make it difficult for the system to identify relevant modules accurately.

### Comparison Across Categories

Keyword queries performed best across all three metrics. Paraphrase and prerequisite queries showed moderate retrieval performance, while vague queries remained the most challenging category.

The difference in performance suggests that retrieval effectiveness depends on how clearly the query expresses the concepts associated with relevant modules. Queries with explicit terminology are easier to match, whereas vague queries provide less information for identifying suitable results.

### Conclusion and Potential Improvements

The evaluation demonstrates that semantic retrieval performs well for keyword-based queries and provides useful results for paraphrase and prerequisite queries. However, its performance decreases substantially for vague queries.

Potential improvements include query expansion, hybrid retrieval combining keyword and semantic methods, and reranking retrieved modules. These approaches could help improve retrieval coverage and ranking quality, particularly for paraphrased, prerequisite-related, and vague queries.

**Note:** These findings are based on 40 evaluation queries, with 10 queries per category. Further evaluation on a larger and more diverse dataset would help determine whether the observed patterns generalize.


In [ ]:
query = "how computers understand language"

# 1. Keyword search
print("=== KEYWORD SEARCH ===")
keyword_results = rag_utils.ranked_keyword_search(query, k=5)

if keyword_results:
    for module in keyword_results:
        print(f"{module['moduleCode']} - {module['title']}")
else:
    print("No keyword matches found.")

# 2. Semantic search
print("\n=== SEMANTIC SEARCH ===")
semantic_results = rag_utils.search(query, k=5)

for module, score in semantic_results:
    print(
        f"{module['moduleCode']} - "
        f"{module['title']} (score: {score:.3f})"
    )

In [ ]:
import pandas as pd

def evaluate_retriever(eval_df, method="semantic", k=5):
    details = []

    for _, row in eval_df.iterrows():
        query = str(row["query"])

        relevant_codes = {
            code.strip().upper()
            for code in str(row["relevant"]).split("|")
            if code.strip()
        }

        # Run selected retriever
        if method == "keyword":
            results = rag_utils.ranked_keyword_search(query, k=k)

            ranked_codes = [
                str(module.get("moduleCode", "")).upper()
                for module in results
            ]

        elif method == "semantic":
            results = rag_utils.search(query, k=k)

            ranked_codes = [
                str(module.get("moduleCode", "")).upper()
                for module, score in results
            ]

        else:
            raise ValueError(
                "method must be 'keyword' or 'semantic'"
            )

        # Rank positions of relevant retrieved modules
        relevant_ranks = [
            rank
            for rank, code in enumerate(ranked_codes, start=1)
            if code in relevant_codes
        ]

        first_relevant_rank = (
            min(relevant_ranks) if relevant_ranks else 0
        )

        # Hit@k: at least one relevant result
        hit_at_k = int(bool(relevant_ranks))

        # Recall@k: relevant retrieved / all labelled relevant
        recall_at_k = (
            len(set(ranked_codes) & relevant_codes)
            / len(relevant_codes)
            if relevant_codes else 0
        )

        # Reciprocal rank of the first relevant result
        reciprocal_rank = (
            1 / first_relevant_rank
            if first_relevant_rank else 0
        )

        details.append({
            "method": method,
            "query": query,
            "category": row["category"],
            "relevant_codes": " | ".join(sorted(relevant_codes)),
            "top_k_module_codes": " | ".join(ranked_codes),
            f"hit_at_{k}": hit_at_k,
            f"recall_at_{k}": recall_at_k,
            "reciprocal_rank": reciprocal_rank,
            "rank_of_first_relevant": first_relevant_rank
        })

    details_df = pd.DataFrame(details)

    summary = {
        "method": method,
        "n_queries": len(details_df),
        f"hit_at_{k}": details_df[f"hit_at_{k}"].mean(),
        f"recall_at_{k}": details_df[f"recall_at_{k}"].mean(),
        "mrr": details_df["reciprocal_rank"].mean()
    }

    return summary, details_df

In [ ]:
keyword_summary, keyword_details = evaluate_retriever(
    eval_df,
    method="keyword",
    k=5
)

semantic_summary, semantic_details = evaluate_retriever(
    eval_df,
    method="semantic",
    k=5
)

comparison = pd.DataFrame([
    keyword_summary,
    semantic_summary
])

comparison = comparison.rename(columns={
    "method": "Method",
    "n_queries": "Queries",
    "hit_at_5": "Hit@5",
    "recall_at_5": "Recall@5",
    "mrr": "MRR"
})

print("OVERALL RETRIEVAL COMPARISON")
display(comparison.round(3))

In [ ]:
all_details = pd.concat(
    [keyword_details, semantic_details],
    ignore_index=True
)

classwise_comparison = (
    all_details
    .groupby(["category", "method"])
    .agg(
        queries=("query", "count"),
        hit_at_5=("hit_at_5", "mean"),
        recall_at_5=("recall_at_5", "mean"),
        mrr=("reciprocal_rank", "mean")
    )
    .reset_index()
)

classwise_comparison = classwise_comparison.rename(columns={
    "category": "Query Category",
    "method": "Method",
    "queries": "Queries",
    "hit_at_5": "Hit@5",
    "recall_at_5": "Recall@5",
    "mrr": "MRR"
})

print("CLASS-WISE RETRIEVAL COMPARISON")
display(classwise_comparison.round(3))

## Interpretation of Retrieval Evaluation Results

### 1. Comparison by Query Category

The evaluation compares keyword-based retrieval and semantic retrieval across four query categories: keyword, paraphrase, prerequisite, and vague queries. The performance is measured using Hit@5, Recall@5, and Mean Reciprocal Rank (MRR).

| Query Category | Method | Hit@5 | Recall@5 | MRR |
|---|---|---:|---:|---:|
| Keyword | Keyword | 1.000 | 0.750 | 0.673 |
| Keyword | Semantic | 1.000 | 0.675 | 0.670 |
| Paraphrase | Keyword | 0.100 | 0.033 | 0.033 |
| Paraphrase | Semantic | 0.600 | 0.400 | 0.392 |
| Prerequisite | Keyword | 0.400 | 0.092 | 0.283 |
| Prerequisite | Semantic | 0.600 | 0.342 | 0.425 |
| Vague | Keyword | 0.100 | 0.100 | 0.050 |
| Vague | Semantic | 0.200 | 0.150 | 0.200 |

### 2. Overall Performance

Since each query category contains 10 queries, the overall metrics can be calculated by taking the average of the four category-level scores.

| Metric | Keyword Retrieval | Semantic Retrieval |
|---|---:|---:|
| Hit@5 | 0.400 | 0.600 |
| Recall@5 | 0.244 | 0.392 |
| MRR | 0.260 | 0.422 |

Semantic retrieval achieves better overall performance, with Hit@5 increasing from 40% to 60%. It also achieves higher Recall@5 and MRR, indicating that it retrieves more relevant modules and tends to rank relevant results higher.

### 3. Observations

- **Keyword queries:** Both methods achieve a Hit@5 of 1.000, meaning that at least one relevant module is retrieved in the top five for every query. Keyword retrieval has slightly higher Recall@5 (0.750 vs. 0.675), while their MRR values are nearly identical.
- **Paraphrase queries:** Semantic retrieval performs substantially better, achieving a Hit@5 of 0.600 compared with 0.100 for keyword retrieval. This demonstrates its ability to retrieve relevant modules when queries use different wording from the module descriptions.
- **Prerequisite queries:** Semantic retrieval achieves a Hit@5 of 0.600, compared with 0.400 for keyword retrieval. It also achieves higher Recall@5 and MRR, suggesting that semantic similarity helps identify conceptually related modules.
- **Vague queries:** Both methods perform poorly. Semantic retrieval achieves a Hit@5 of 0.200, while keyword retrieval achieves 0.100. This suggests that queries with insufficient detail remain challenging for both approaches.

### 4. Conclusion 

The results indicate that semantic retrieval is a stronger baseline for the NUSMods RAG system, particularly for paraphrase and prerequisite queries. However, keyword retrieval remains competitive for queries containing explicit terms that match module titles and descriptions.

**YOUR TURN:** pick a query from the table above that missed (`hit_at_5` is 0), or ask your own question, and look at what the retriever returned instead.

In [ ]:
# YOUR TURN: change the question, then re-run
query = "Which NUS modules cover attention mechanisms?"

results = rag_utils.search(query, k=5)

print(f"Query: {query}\n")
print("Top 5 retrieved modules:\n")

for rank, (module, score) in enumerate(results, start=1):
    print(f"{rank}. {module.get('moduleCode', 'N/A')} - "
          f"{module.get('title', 'N/A')}")
    print(f"   Similarity score: {score:.4f}\n")

## Recap

- **Generation:** the retrieved modules go into the prompt as context, and the instructions keep Gemini to that context and make it cite module codes.
- **Evaluation:** Hit@5 asks whether a relevant module made the top 5; MRR rewards finding it near the top.
- **The weak link is retrieval:** with the wrong context, even a good LLM gives a poor answer.